In [1]:
from dotenv import load_dotenv
from src.datasources import glofas
from src.utils import blob

load_dotenv()

/Users/hannahker/Desktop/AA/ds-aa-nga-flooding/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
STATION_NAME = "makurdi"
PRODUCT_TYPE = "ensemble"  # or "control"

## Check raw blobs

In [3]:
raw_prefix = (
    f"ds-aa-nga-flooding/raw/glofas/reforecast/"
    f"glofas_raw_reforecast_{STATION_NAME}_{PRODUCT_TYPE}_"
)
raw_blobs = sorted(
    x for x in blob.list_container_blobs(name_starts_with=raw_prefix)
    if x.endswith(".grib")
)
print(f"Found {len(raw_blobs)} raw GRIB files")
for b in raw_blobs:
    print(" ", b)

Found 84 raw GRIB files
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2003_lt120-192.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2003_lt216-288.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2003_lt24-96.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2003_lt312-384.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2004_lt120-192.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2004_lt216-288.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2004_lt24-96.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2004_lt312-384.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforecast_makurdi_ensemble_2005_lt120-192.grib
  ds-aa-nga-flooding/raw/glofas/reforecast/glofas_raw_reforeca

## Process and upload

Reads each GRIB, extracts `dis24` at the station grid point across all leadtimes and ensemble members, concatenates, and uploads to:
`ds-aa-nga-flooding/processed/glofas/glofas_reforecast_{station}_{product_type}.parquet`

In [4]:
glofas.process_glofas_reforecast(STATION_NAME, PRODUCT_TYPE)

100%|██████████| 84/84 [03:21<00:00,  2.40s/it]


## Verify output

In [6]:
df = glofas.load_glofas_reforecast(STATION_NAME, PRODUCT_TYPE)
print(f"Rows:        {len(df):,}")
print(f"Columns:     {list(df.columns)}")
print(f"Leadtimes:   {sorted(df['leadtime'].unique())}")
if 'number' in df.columns:
    print(f"Ens members: {sorted(df['number'].unique())}")
print(f"Time range:  {df['time'].min().date()} → {df['time'].max().date()}")
df.head()

Rows:        358,400
Columns:     ['number', 'time', 'valid_time', 'dis24', 'leadtime']
Leadtimes:   [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)]
Ens members: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Time range:  2003-06-01 → 2023-11-25


,number,time,valid_time,dis24,leadtime
0,1,2003-06-01,2003-06-02,1674.87500,1
1,2,2003-06-01,2003-06-02,1671.40625,1
2,3,2003-06-01,2003-06-02,1675.00000,1
3,4,2003-06-01,2003-06-02,1675.96875,1
4,5,2003-06-01,2003-06-02,1670.25000,1
